In [1]:
!nvidia-smi
%cd /home/benle1/unlearning-analysis

Mon May 11 17:39:33 2026       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 545.23.08              Driver Version: 545.23.08    CUDA Version: 12.3     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  Tesla V100-PCIE-16GB           On  | 00000000:AF:00.0 Off |                    0 |
| N/A   83C    P0             116W / 250W |  13730MiB / 16384MiB |     99%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

In [2]:
from typing import List, Literal, Tuple, Dict
import pandas as pd
import numpy as np
from vision_unlearning.benchmarks.I_care import get_interference_per_entity_path
from vision_unlearning.datasets.testbed import plot_heatmap
from vision_unlearning.utils.logger import get_logger, setup_loggers
from vision_unlearning.benchmarks.I_care import analyze_relationship_numerical, analyze_relationship_categorical, analyze_correlation_between_pairwise_metrics, exists_interference_per_pair

import os

import dotenv
dotenv.load_dotenv()

assert len(os.getenv('HF_TOKEN'))>0



from vision_unlearning.benchmarks.I_care import get_interference_per_pair_path, get_interference_per_pair
from vision_unlearning.datasets.testbed import get_metadata_filtered, get_similarity_clip_df

from vision_unlearning.benchmarks.I_care import check_eval_results
from vision_unlearning.datasets.testbed import get_target_overwrite, get_generated_dataset_folder, get_unlearned_model_folder, exists_unlearned_model, exists_unlearned_dataset, exists_metadata_filtered, get_metadata_filtered_path, task_to_dataset_map
from vision_unlearning.utils.data_generation import generate_dataset

from vision_unlearning.integrations.huggingface import huggingface_dataset_upload, huggingface_dataset_file_upload

task_list: List[Literal['scenes', 'objects', 'breeds', 'people']] = ['people', 'scenes', 'breeds']

index_start_list: List[int] = [0, 0, 0]  # for each task
max_identities_list: List[int] = [100, 100, 100]  # for each task


method_list: List[List[Literal['munba', 'uce', 'distil']]] = [  # for each task
    ['uce', 'munba', 'distil'],
    ['uce', 'munba', 'distil'],
    ['uce', 'munba', 'distil'],
]
num_train_epochs_list: List[List[int]] = [
    [0, 200, 400],
    [0, 200, 100],
    [0, 0, 0],
]# for each task, then for each method in the same order

generate_dataset_seeds = [42, 43, 44, 45]  # applied to all runs

#####


base_folder = 'assets'
hf_repository_name = 'LeonardoBenitez/VisionUnlearningEvaluationTestbeds'
upload: bool = True  # If false, this notebook serves just as completion status check... TODO maybe these two things should be separated...


logger = get_logger('unlearning_analysis')
setup_loggers(modules_info=['unlearning'])

2026-05-11 17:39:53.090353: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778513993.124573 1383945 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778513993.135738 1383945 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778513993.162462 1383945 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778513993.162480 1383945 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778513993.162484 1383945 computation_placer.cc:177] computation placer alr

# Upload

In [3]:
#[e['name'] for e in get_metadata_filtered('scenes', base_folder=base_folder)]

In [4]:
# metadata
completion_status_metadata: Dict[Literal['scenes', 'objects', 'breeds', 'people'], int] = {}
for t, task in enumerate(task_list):
    exists: bool = exists_metadata_filtered(task, base_folder=base_folder)
    completion_status_metadata[task] = 1 if exists else 0
    print(f"{t} ({task}) -> {exists}")
    if upload and exists:
        huggingface_dataset_file_upload(
            file_path=get_metadata_filtered_path(task, base_folder=base_folder),
            dataset_repository=hf_repository_name,
            dataset_path=get_metadata_filtered_path(task, base_folder=''),
            token=os.getenv('HF_TOKEN'),
        )
print('-'*50)
print("Completion Status - Metadata:")
df_status_metadata = pd.DataFrame.from_dict(completion_status_metadata, orient='index', columns=['metadata_filtered_percentage'])
df_status_metadata['metadata_filtered_percentage'] = df_status_metadata['metadata_filtered_percentage'] * 100
df_status_metadata.style.background_gradient(cmap='RdYlGn', vmin=0, vmax=100).format("{:.0f}%")

0 (people) -> True


No files have been modified since last commit. Skipping to prevent empty commit.


1 (scenes) -> True


No files have been modified since last commit. Skipping to prevent empty commit.


2 (breeds) -> True


No files have been modified since last commit. Skipping to prevent empty commit.


--------------------------------------------------
Completion Status - Metadata:


,metadata_filtered_percentage
people,100%
scenes,100%
breeds,100%


In [5]:
# Training dataset
completion_status_datasets: Dict[Literal['scenes', 'objects', 'breeds', 'people'], int] = {}
for t, task in enumerate(task_list):
    dataset_base_path = task_to_dataset_map[task]
    exists: bool = os.path.exists(os.path.join('assets', dataset_base_path))
    completion_status_datasets[task] = 1 if exists else 0
    print(f"{t} ({task}) -> {exists}")
    if False and upload and exists:  # Datasets dont need to be uploaded because they can be redownloaded. Also, `people` dont use symlinks...
        huggingface_dataset_upload(
            folder_datasets = 'assets',
            dataset_repository=hf_repository_name,
            dataset_config=dataset_base_path,
            token=os.getenv('HF_TOKEN'),
        )
        break
print('-'*50)
print("Completion Status - Training Datasets:")
df_status_datasets = pd.DataFrame.from_dict(completion_status_datasets, orient='index', columns=['training_dataset_percentage'])
df_status_datasets['training_dataset_percentage'] = df_status_datasets['training_dataset_percentage'] * 100
df_status_datasets.style.background_gradient(cmap='RdYlGn', vmin=0, vmax=100).format("{:.0f}%")

0 (people) -> True
1 (scenes) -> True
2 (breeds) -> True
--------------------------------------------------
Completion Status - Training Datasets:


,training_dataset_percentage
people,100%
scenes,100%
breeds,100%


In [7]:
# Unlearned models
completion_status_interference_pair: Dict[Literal['scenes', 'objects', 'breeds', 'people'], Dict[Literal['munba', 'uce', 'distil'], int]] = {}
for t, task in enumerate(task_list):
    metadata_filtered = get_metadata_filtered(task, base_folder=base_folder)
    index_start = index_start_list[t]
    max_identities = max_identities_list[t]
    completion_status_interference_pair[task] = {}
    for m, method in enumerate(method_list[t]):
        num_train_epochs = num_train_epochs_list[t][m]
        completion_status_interference_pair[task][method] = 0
        for index in range(index_start, index_start + max_identities):
            target = metadata_filtered[index]['name']
            exists: bool = exists_unlearned_model(task, method, num_train_epochs, target, base_folder=base_folder)
            print(f"t={t} ({task}), m={m} ({method}), index={index} ({target}) -> " + ("\033[92m DONE \033[0m" if exists else "\033[91m NO... \033[0m"))
            completion_status_interference_pair[task][method] += 1 if exists else 0
            already_uploaded = huggingface_dataset_exists(  # TODO this is always returnign false!
                dataset_repository = hf_repository_name,
                dataset_config = get_unlearned_model_folder(task, method, num_train_epochs, target, base_folder=''),
                token = os.getenv('HF_TOKEN'),
            )
            if upload and exists and not already_uploaded:                
            #if False:
                huggingface_dataset_upload(
                    folder_datasets = 'assets',
                    dataset_repository = hf_repository_name,
                    dataset_config = get_unlearned_model_folder(task, method, num_train_epochs, target, base_folder=''),
                    token = os.getenv('HF_TOKEN'),
                )
        completion_status_interference_pair[task][method] = int(completion_status_interference_pair[task][method] / max_identities * 100)
        #break
    #break

print('-'*50)
print("Completion Status - Unlearned Models:")
df_status_models = pd.DataFrame.from_dict(completion_status_interference_pair, orient='index')

all_methods = list(dict.fromkeys(method for methods in method_list for method in methods))
df_status_models = df_status_models.reindex(columns=all_methods).fillna(0).astype(int)

df_status_models.style.background_gradient(cmap='RdYlGn', vmin=0, vmax=100).format("{:.0f}%")

            

t=0 (people), m=0 (uce), index=0 (George_W_Bush) ->  DONE 
t=0 (people), m=0 (uce), index=1 (Colin_Powell) ->  DONE 
t=0 (people), m=0 (uce), index=2 (Tony_Blair) ->  DONE 
t=0 (people), m=0 (uce), index=3 (Donald_Rumsfeld) ->  DONE 
t=0 (people), m=0 (uce), index=4 (Ariel_Sharon) ->  DONE 
t=0 (people), m=0 (uce), index=5 (Junichiro_Koizumi) ->  DONE 
t=0 (people), m=0 (uce), index=6 (John_Ashcroft) ->  DONE 
t=0 (people), m=0 (uce), index=7 (Jacques_Chirac) ->  DONE 
t=0 (people), m=0 (uce), index=8 (Serena_Williams) ->  DONE 
t=0 (people), m=0 (uce), index=9 (Vladimir_Putin) ->  DONE 
t=0 (people), m=0 (uce), index=10 (Gloria_Macapagal_Arroyo) ->  DONE 
t=0 (people), m=0 (uce), index=11 (Arnold_Schwarzenegger) ->  DONE 
t=0 (people), m=0 (uce), index=12 (Jennifer_Capriati) ->  DONE 
t=0 (people), m=0 (uce), index=13 (Lleyton_Hewitt) ->  DONE 
t=0 (people), m=0 (uce), index=14 (Laura_Bush) ->  DONE 
t=0 (people), m=0 (uce), index=15 (Alejandro_Toledo) ->  DONE 
t=0 (people), m=0 (uce

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

t=1 (scenes), m=0 (uce), index=81 (outcropping) ->  DONE 


Processing Files (0 / 0): |          |  0.00B /  0.00B            

...

New Data Upload: |          |  0.00B /  0.00B            

t=1 (scenes), m=2 (distil), index=98 (office_building) ->  DONE 


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

HfHubHTTPError: 429 Client Error: Too Many Requests for url: https://huggingface.co/api/datasets/LeonardoBenitez/VisionUnlearningEvaluationTestbeds/commit/main (Request ID: Root=1-69ff4dd8-44b0714c7f5415130aad4bbb;d637ae4a-89af-4918-a0ce-4967b36315b9)

You have exceeded the rate limit for repository commits (128 per hour). You can retry this action in about 1 hour. To reduce the number of commits, you can upload entire folders at once using the Hub Python library: https://huggingface.co/docs/huggingface_hub/guides/upload#upload-a-folder or for large folders: https://huggingface.co/docs/huggingface_hub/guides/upload#upload-a-large-folder. To increase your rate limit for this action, you can upgrade to a paid plan at https://huggingface.co/pricing.

In [5]:
%%capture
%%time

# Generated datasets
completion_status_gen: Dict[Literal['scenes', 'objects', 'breeds', 'people'], Dict[Literal['munba', 'uce', 'distil'], int]] = {}
for t, task in enumerate(task_list):
    metadata_filtered = get_metadata_filtered(task, base_folder=base_folder)
    index_start = index_start_list[t]
    max_identities = max_identities_list[t]
    completion_status_gen[task] = {}
    for m, method in enumerate(method_list[t]):
        num_train_epochs = num_train_epochs_list[t][m]
        completion_status_gen[task][method] = 0
        for index in range(index_start, index_start + max_identities):
            target = metadata_filtered[index]['name']
            target_preprocessed, target_overwrite = get_target_overwrite(task, method, target)
            prompts = [f"An image of {get_target_overwrite(task, method, m['name'])[0]}" for m in metadata_filtered]
            generated_dataset_output_path = get_generated_dataset_folder(task, method, num_train_epochs, target_preprocessed, base_folder=base_folder)
            exists: bool = exists_unlearned_dataset(generated_dataset_output_path, generate_dataset_seeds, prompts)
            print(f"t={t} ({task}), m={m} ({method}), index={index} ({target}) -> " + ("\033[92m DONE \033[0m" if exists else "\033[91m NO... \033[0m"))
            completion_status_gen[task][method] += 1 if exists else 0
            already_uploaded = huggingface_dataset_exists(
                dataset_repository = hf_repository_name,
                dataset_config = get_generated_dataset_folder(task, method, num_train_epochs, target_preprocessed, base_folder=''),
                token = os.getenv('HF_TOKEN'),
            )
            if upload and exists and not already_uploaded:                
            #if False:
                huggingface_dataset_upload(
                    folder_datasets = 'assets',
                    dataset_repository = hf_repository_name,
                    dataset_config = get_generated_dataset_folder(task, method, num_train_epochs, target_preprocessed, base_folder=''),
                    token = os.getenv('HF_TOKEN'),
                )
                #break
        completion_status_gen[task][method] = int(completion_status_gen[task][method] / max_identities * 100)
        #break
    #break


It seems you are trying to upload a large folder at once. This might take some time and then fail if the folder is too large. For such cases, it is recommended to upload in smaller batches or to use `HfApi().upload_large_folder(...)`/`hf upload-large-folder` instead. For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/upload#upload-a-large-folder.
It seems you are trying to upload a large folder at once. This might take some time and then fail if the folder is too large. For such cases, it is recommended to upload in smaller batches or to use `HfApi().upload_large_folder(...)`/`hf upload-large-folder` instead. For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/upload#upload-a-large-folder.
It seems you are trying to upload a large folder at once. This might take some time and then fail if the folder is too large. For such cases, it is recommended to upload in smaller batches or to use `HfApi().upload_large_folder(..

In [6]:
print('-'*50)
print("Completion Status - Generated Datasets:")
df_status_models = pd.DataFrame.from_dict(completion_status_gen, orient='index')

all_methods = list(dict.fromkeys(method for methods in method_list for method in methods))
df_status_models = df_status_models.reindex(columns=all_methods).fillna(0).astype(int)

df_status_models.style.background_gradient(cmap='RdYlGn', vmin=0, vmax=100).format("{:.0f}%")

--------------------------------------------------
Completion Status - Generated Datasets:


,uce,munba,distil
people,100%,54%,100%
scenes,100%,0%,100%
breeds,0%,0%,0%


In [ ]:
'''
task = 'people'
method = 'distil'
num_train_epochs = 400


metadata_filtered = get_metadata_filtered(task, base_folder=base_folder)

index=57
target = metadata_filtered[index]['name']
target_preprocessed, target_overwrite = get_target_overwrite(task, method, target)
prompts = [f"An image of {get_target_overwrite(task, method, m['name'])[0]}" for m in metadata_filtered]
print(target_preprocessed)
generated_dataset_output_path = get_generated_dataset_folder(task, method, num_train_epochs, target_preprocessed, base_folder=base_folder)

exists_unlearned_dataset(generated_dataset_output_path, generate_dataset_seeds, prompts)
#get_generated_dataset_folder(task, method, num_train_epochs, target_preprocessed, base_folder=base_folder)
'''

In [ ]:
# Embeddings
# TODO

In [ ]:
# 
# interference_per_pair

In [ ]:
get_interference_per_pair_path(task, index, method, num_train_epochs, base_folder='')

In [ ]:
# Interpefernece per pair
completion_status_interference_pair: Dict[Literal['scenes', 'objects', 'breeds', 'people'], Dict[Literal['munba', 'uce', 'distil'], int]] = {}
for t, task in enumerate(task_list):
    metadata_filtered = get_metadata_filtered(task, base_folder=base_folder)
    index_start = index_start_list[t]
    max_identities = max_identities_list[t]
    completion_status_interference_pair[task] = {}
    for m, method in enumerate(method_list[t]):
        num_train_epochs = num_train_epochs_list[t][m]
        completion_status_interference_pair[task][method] = 0
        for index in range(index_start, index_start + max_identities):
            target = metadata_filtered[index]['name']
            exists: bool = exists_interference_per_pair(task, index, method, num_train_epochs, base_folder=base_folder)
            print(f"t={t} ({task}), m={m} ({method}), index={index} ({target}) -> " + ("\033[92m DONE \033[0m" if exists else "\033[91m NO... \033[0m"))
            completion_status_interference_pair[task][method] += 1 if exists else 0
            already_uploaded = huggingface_dataset_file_exists(
                dataset_repository = hf_repository_name,
                dataset_path = get_interference_per_pair_path(task, index, method, num_train_epochs, base_folder=''),
                token = os.getenv('HF_TOKEN'),
            )
            if upload and exists and already_uploaded:
            #if False:
                huggingface_dataset_file_upload(
                    file_path=get_interference_per_pair_path(task, index, method, num_train_epochs, base_folder=base_folder),
                    dataset_repository=hf_repository_name,
                    dataset_path=get_interference_per_pair_path(task, index, method, num_train_epochs, base_folder=''),
                    token=os.getenv('HF_TOKEN'),
                )
                #break
        completion_status_interference_pair[task][method] = int(completion_status_interference_pair[task][method] / max_identities * 100)
        #break
    #break

print('-'*50)
print("Completion Status - Interference per Pair:")
df_status_models = pd.DataFrame.from_dict(completion_status_interference_pair, orient='index')

all_methods = list(dict.fromkeys(method for methods in method_list for method in methods))
df_status_models = df_status_models.reindex(columns=all_methods).fillna(0).astype(int)

df_status_models.style.background_gradient(cmap='RdYlGn', vmin=0, vmax=100).format("{:.0f}%")

            

In [ ]:
# interference_per_entity
completion_status_interference_entity: Dict[type_task, int] = {}


for t, task in enumerate(task_list):
    p = get_interference_per_entity_path(task)
    exists: bool = os.path.exists(p)
    completion_status_interference_entity[task] = 1 if exists else 0
    print(f"t={t} ({task}) -> {exists}")
    already_uploaded = huggingface_dataset_file_exists(
        dataset_repository = hf_repository_name,
        dataset_path = get_interference_per_entity_path(task, base_folder=''),
        token = os.getenv('HF_TOKEN'),
    )
    if upload and exists and not already_uploaded:
        huggingface_dataset_file_upload(
            file_path=get_interference_per_entity_path(task, base_folder=base_folder),
            dataset_repository=hf_repository_name,
            dataset_path=get_interference_per_entity_path(task, base_folder=''),
            token=os.getenv('HF_TOKEN'),
        )

print('-'*50)
print("Completion Status - Interference per Entity:")
df_status_interference_entity = pd.DataFrame.from_dict(completion_status_interference_entity, orient='index', columns=['interference_per_entity_percentage'])
df_status_interference_entity['interference_per_entity_percentage'] = df_status_interference_entity['interference_per_entity_percentage'] * 100
df_status_interference_entity.style.background_gradient(cmap='RdYlGn', vmin=0, vmax=100).format("{:.0f}%")



--------------------------------------------------
Completion Status - Interference per Entity:


,interference_per_entity_percentage
people,100%
scenes,100%
breeds,100%


In [ ]:
# Outputs of the Result Templates
# Alredy uploaded as they are computed, see `5c. Run all RTs.py`
from vision_unlearning.benchmarks.I_care import ResultTemplateSimilarityMatrix
rt_list = [ResultTemplateSimilarityMatrix]
completion_status_result: Dict[str, int] = {}
for rt_class in rt_list:
    if rt_class == ResultTemplateSimilarityMatrix:
        completion_status_result[rt_class.__name__] = 0.0
        max_count = 0
        for task in task_list:
            for similarity_metric in ['clip']:
                rt = rt_class(task=task, similarity_metric=similarity_metric)
                max_count += 1
                exists: bool = os.path.exists(rt._get_data_path_local())
                if exists:
                    print(f"RT={rt.__class__.__name__}, task={task}, similarity_metric={similarity_metric} -> " + "\033[92m DONE \033[0m")
                    completion_status_result[rt.__class__.__name__] += 1
                    if upload:
                        huggingface_dataset_file_upload(
                            file_path=rt._get_data_path_local(),
                            dataset_repository=hf_repository_name,
                            dataset_path=rt._get_data_path_remote(),
                            token=os.getenv('HF_TOKEN'),
                        )
        completion_status_result[rt_class.__name__] = int(completion_status_result[rt_class.__name__] / max_count * 100)
                


            
print('-'*50)
print("Completion Status - Result Templates:")
pd.DataFrame.from_dict(completion_status_result, orient='index', columns=['Completion Status'])\
    .style.background_gradient(cmap='RdYlGn', vmin=0, vmax=100).format("{:.0f}%")


RT=ResultTemplateSimilarityMatrix, task=people, similarity_metric=clip ->  DONE 
RT=ResultTemplateSimilarityMatrix, task=scenes, similarity_metric=clip ->  DONE 
RT=ResultTemplateSimilarityMatrix, task=breeds, similarity_metric=clip ->  DONE 
--------------------------------------------------
Completion Status - Result Templates:


,Completion Status
ResultTemplateSimilarityMatrix,100%


In [10]:
rt.__class__.__name__

'ResultTemplateSimilarityMatrix'

In [8]:
ResultTemplateSimilarityMatrix.__name__

'ResultTemplateSimilarityMatrix'

In [ ]:


    metadata_filtered = get_metadata_filtered(task, base_folder=base_folder)
    index_start = index_start_list[t]
    max_identities = max_identities_list[t]
    completion_status_interference_pair[task] = {}
    for m, method in enumerate(method_list[t]):
        num_train_epochs = num_train_epochs_list[t][m]
        completion_status_interference_pair[task][method] = 0
        for index in range(index_start, index_start + max_identities):
            target = metadata_filtered[index]['name']
            exists: bool = exists_interference_per_pair(task, index, method, num_train_epochs, base_folder=base_folder)
            print(f"t={t} ({task}), m={m} ({method}), index={index} ({target}) -> " + ("\033[92m DONE \033[0m" if exists else "\033[91m NO... \033[0m"))
            completion_status_interference_pair[task][method] += 1 if exists else 0
            if upload and exists:
            #if False and upload and exists:
                huggingface_dataset_file_upload(
                    file_path=get_interference_per_pair_path(task, index, method, num_train_epochs, base_folder=base_folder),
                    dataset_repository=hf_repository_name,
                    dataset_path=get_interference_per_pair_path(task, index, method, num_train_epochs, base_folder=''),
                    token=os.getenv('HF_TOKEN'),
                )
                #break
        completion_status_interference_pair[task][method] = int(completion_status_interference_pair[task][method] / max_identities * 100)
        #break
    #break
